#                            🟨 **Make a RAG System**🟨

# 🧰 **Utils & Imports**

In [1]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEndpointEmbeddings
from langchain_chroma import Chroma
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
from langchain_community.document_loaders import PyPDFLoader

In [2]:
loader = PyPDFLoader("Monology.pdf")
documents = loader.load()
print(len(documents))
print(documents[0].page_content)

1
اکنون زمان آن است که برخیزیم و با دیدگانی گشوده به پهنای قطر تنهاییمان کمی نقد تر گذران کنیم.دیدن کافی است.در لحظه ای که 
ذرهای به آیندهی ناشناتخه خود واقف نیستیم چگونه امید را به سراسر پیکر بی جان خود تزریق کنیم.  
دیدن کافی است.گاهی پس از سالیان متمادی همچنان در اکناف خود جزییاتی برای مشاهده جلب نظر میکند که در واقع یک عمر قافل 
از آن بودیم.خوب دیدن احتیاج به تمرین دارد.خوب دیدن افرادی که به گمانمان ابدی اند.خوب دیدن شرایطی که به پندارمان و به 
بهانه ی استمرار آن،نسبت به اشتیاق در لحظه بودن آن فلج هیجانی شده ایم.  
سرزنش کردن خویش را به مثابه یک کودکی که هیچگاه خود را ملامت نمیکند کنار بگذار و کمی وقت بخر. 
وقت بخر برای عزلت خویش که سوداگریی به غایت پر سود است رفیق.کمی رستگاری در این ایام پایانی نسخه است.  
علاجش همین است.ایام پایانی.بهتر است این عبارت مانند برخورد شهاب سنگی تمامییتت را دگرگون سازد تا این که بخواهی یک 
عمر در کمال مذبوحانه زیستن خودت را مستهلک کنی.اگر این را آونگ روحت کنی کمتر دچار محنت خواهی شد.کمتر به ساحت  
احدیت استعانت برای جهان ناسوت می ورزی.کمتر مفتون عالم ب

# 🔠 **Chuncking**

In [3]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=80)
chunks = text_splitter.split_documents(documents)

print("Number of chunks:", len(chunks))
for i, chunk in enumerate(chunks):
    print(f"\n--- Chunk {i + 1} ---")
    print(chunk.page_content)

Number of chunks: 4

--- Chunk 1 ---
اکنون زمان آن است که برخیزیم و با دیدگانی گشوده به پهنای قطر تنهاییمان کمی نقد تر گذران کنیم.دیدن کافی است.در لحظه ای که 
ذرهای به آیندهی ناشناتخه خود واقف نیستیم چگونه امید را به سراسر پیکر بی جان خود تزریق کنیم.  
دیدن کافی است.گاهی پس از سالیان متمادی همچنان در اکناف خود جزییاتی برای مشاهده جلب نظر میکند که در واقع یک عمر قافل 
از آن بودیم.خوب دیدن احتیاج به تمرین دارد.خوب دیدن افرادی که به گمانمان ابدی اند.خوب دیدن شرایطی که به پندارمان و به

--- Chunk 2 ---
بهانه ی استمرار آن،نسبت به اشتیاق در لحظه بودن آن فلج هیجانی شده ایم.  
سرزنش کردن خویش را به مثابه یک کودکی که هیچگاه خود را ملامت نمیکند کنار بگذار و کمی وقت بخر. 
وقت بخر برای عزلت خویش که سوداگریی به غایت پر سود است رفیق.کمی رستگاری در این ایام پایانی نسخه است.  
علاجش همین است.ایام پایانی.بهتر است این عبارت مانند برخورد شهاب سنگی تمامییتت را دگرگون سازد تا این که بخواهی یک 
عمر در کمال مذبوحانه زیستن خودت را مستهلک کنی.اگر این را آونگ روحت کنی کمتر دچار محنت خواهی شد.کمتر به ساحت

--- C

# 🔀 **Embedding**

In [4]:
HF_API_KEY = "Your_API_Key"
embeddings = HuggingFaceEndpointEmbeddings(model="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", huggingfacehub_api_token=HF_API_KEY)

query = "کل مظنه عالم چیست؟"
vector = embeddings.embed_query(query)                   # Convert Query to Vector  
print("Vector dimension:", len(vector))
print("First 10 values:", vector[:10])

Vector dimension: 384
First 10 values: [0.07877548784017563, 0.12519089877605438, -0.21802282333374023, -0.1692856103181839, 0.6267856359481812, -0.012266772799193859, 0.3091687262058258, 0.043411001563072205, 0.1119406521320343, -0.11941393464803696]


# 📗 **Create ChromaDB**

In [8]:
vectorstore = Chroma.from_documents(documents=chunks ,embedding=embeddings ,collection_name='monolog' ,persist_directory="monologDB",collection_metadata={"hnsw:space": "cosine"})        # persist_directory :: save VectorDB    # collection_name ::DB Name   # collection_metadata :: for score_threshold 0.7 in retriever
vectorstore = Chroma(collection_name='monolog' ,persist_directory="monologDB" ,embedding_function=embeddings )                               # load VectorDB

collection = vectorstore._collection
data = collection.get()

print(len(data["documents"]))          # No. documnets
for text in data["documents"]:         # texts 0.6
    print("================")
    print(text)

# data["metadatas"]

4
اکنون زمان آن است که برخیزیم و با دیدگانی گشوده به پهنای قطر تنهاییمان کمی نقد تر گذران کنیم.دیدن کافی است.در لحظه ای که 
ذرهای به آیندهی ناشناتخه خود واقف نیستیم چگونه امید را به سراسر پیکر بی جان خود تزریق کنیم.  
دیدن کافی است.گاهی پس از سالیان متمادی همچنان در اکناف خود جزییاتی برای مشاهده جلب نظر میکند که در واقع یک عمر قافل 
از آن بودیم.خوب دیدن احتیاج به تمرین دارد.خوب دیدن افرادی که به گمانمان ابدی اند.خوب دیدن شرایطی که به پندارمان و به
بهانه ی استمرار آن،نسبت به اشتیاق در لحظه بودن آن فلج هیجانی شده ایم.  
سرزنش کردن خویش را به مثابه یک کودکی که هیچگاه خود را ملامت نمیکند کنار بگذار و کمی وقت بخر. 
وقت بخر برای عزلت خویش که سوداگریی به غایت پر سود است رفیق.کمی رستگاری در این ایام پایانی نسخه است.  
علاجش همین است.ایام پایانی.بهتر است این عبارت مانند برخورد شهاب سنگی تمامییتت را دگرگون سازد تا این که بخواهی یک 
عمر در کمال مذبوحانه زیستن خودت را مستهلک کنی.اگر این را آونگ روحت کنی کمتر دچار محنت خواهی شد.کمتر به ساحت
احدیت استعانت برای جهان ناسوت می ورزی.کمتر مفتون عالم با ت

# 🔎 **Retriever / ChromaTest**

In [26]:
# retriever = vectorstore.as_retriever(search_kwargs={"k": 3})                        #score_threshold :: only upper 70%           #k :: only top 3 chunck

retriever = vectorstore.as_retriever(search_type="similarity_score_threshold"
                                    ,search_kwargs={"score_threshold": 0.40,"k": 3})  #score_threshold :: only upper 70%           #k :: only top 3 chunck

query =  """ 
نظر نویسنده راجب تنهایی
 """

query =  """ 
 نظر نویسنده راجب فلسفه
 """

results = retriever.invoke(query)

print("\nRetrieved documents:")
for i, doc in enumerate(results):
    print(f"\n--- Result {i + 1} ---")
    print(doc.page_content)


Retrieved documents:

--- Result 1 ---
احدیت استعانت برای جهان ناسوت می ورزی.کمتر مفتون عالم با تمام تجلیات مرصع اش میشوی.  
آری رفیق ، باکی ندارم که اپیکوریسم را با ایزولاسیونیسم ستد کنم.باکی ندارم در این روزگاری که ناخودآگاه جمعی منفعلانه به 
کیاست منفردانه استیلا گردیده ،من با علم به عوارض آن دگراندیش باشم.  
برخیز و بعد از صعدایی عمیق که تنها مخاطب پژواک آنی، راه زیستن را همچون پارسایی که در عین وارستگی ، به وابستگی های


## **Accuracy Test**

In [27]:
docs = vectorstore.similarity_search_with_relevance_scores(query, k=3)

for doc, score in docs:
    print("RELEVANCE:", score)
    print("TEXT:", doc.page_content[:100])
    print("-" * 50)

RELEVANCE: 0.4357413649559021
TEXT: احدیت استعانت برای جهان ناسوت می ورزی.کمتر مفتون عالم با تمام تجلیات مرصع اش میشوی.  
آری رفیق ، باک
--------------------------------------------------
RELEVANCE: 0.3523501753807068
TEXT: بهانه ی استمرار آن،نسبت به اشتیاق در لحظه بودن آن فلج هیجانی شده ایم.  
سرزنش کردن خویش را به مثابه 
--------------------------------------------------
RELEVANCE: 0.35084909200668335
TEXT: لاهوتی کیهان خود چنگ میزند طی کن که در این اثنا ثمره ای چون کمی فرزانگی و خردی بیشتربهره ات گردد که 
--------------------------------------------------


# 🧠 **LLM**

In [20]:
GROQ_API_KEY = "Your_API_Key"
llm = ChatGroq(model="openai/gpt-oss-20b",api_key=GROQ_API_KEY  ,temperature=0 , max_tokens=1000)      # temperature :: creativity of model (for RAG few is better)
response = llm.invoke("کل مظنه عالم چیست؟")             # test raw model
response.content.replace('\u200c',' ')

'**کل مظنهٔ عالم** (یا «کلّیّتِ جهان») به طور کلی به تمام آنچه وجود دارد اشاره دارد؛ یعنی تمام فضا، زمان، ماده، انرژی، قوانین فیزیکی، رویدادها، و حتی مفاهیم ذهنی و معنوی که در آن جای می گیرند. در ادامه، چند دیدگاه مختلف – علمی، فلسفی، و دینی – را بررسی می کنیم تا تصویری جامع تر از این مفهوم ارائه دهیم.\n\n---\n\n## ۱. دیدگاه علمی\n\n### الف. مدل استاندارد فیزیک\n- **ماده و انرژی**: ماده (ذرات بنیادی، پروتون، نوترون، الکترون) و انرژی (فوتون، انرژی جنبشی، انرژی پتانسیلی).\n- **قوانین فیزیکی**: معادلات گرانش (نيوتن، نسبیت عام)، کوانتوم، و نیروهای بنیادی (نابود، قوی، الکترومغناطیسی، ضعیف).\n- **فضا و زمان**: چهار بعدی (سه فضا + زمان) که در قالب هندسه ی کروم کُر (کُر پوش) توصیف می شود.\n\n### ب. نظریهٔ انفجار بزرگ (Big Bang)\n- **شروع**: حدود 13.8 میلیارد سال پیش، یک نقطه ی بی نهایت چگال و گرم که به صورت انفجاری گسترش یافت.\n- **گسترش مداوم**: جهان همچنان در حال گسترش است؛ سرعت گسترش با زمان تغییر می کند (تسریع گسترش به دلیل ماده تاریک).\n\n### ج. ساختار کائنات\n- **سیستم های نجومی**: ستارگ

# 🧑 **Prompt Template**

In [28]:
def format_docs(docs):                                           # convert documnet(docs) to text
    return "\n\n".join(doc.page_content for doc in docs)

In [29]:
prompt = ChatPromptTemplate.from_template("""
تو یک دستیار هوشمند هستی که فقط بر اساس Context پاسخ می‌دهد.

قوانین:

1. فقط از اطلاعات موجود در Context استفاده کن.
2. هیچ اطلاعاتی را از دانش قبلی خودت اضافه نکن.
3. اگر پاسخ سؤال به طور کامل یا کافی در Context وجود ندارد،
   بگو: «اطلاعات کافی در اختیار ندارم.»
4. پاسخ را با لحن و ادبیات خود متن بده.
5. لحن و ادبیات متن اصلی را حفظ کن.
6. اگر پاسخ شامل چند نکته است، آنها را جداگانه بیان کن.

Context:
{context}

Question:
{question}

Answer: """)

# 📉**RAG Chain**

In [30]:
rag_chain = ({"context": retriever | format_docs, "question": RunnablePassthrough() } | prompt | llm | StrOutputParser())

# ✅**Prompt**

In [33]:
user_prompt = '''
نظر نویسنده راجب فلسفه
 '''

answer = rag_chain.invoke(user_prompt)
print(answer)

نظری که نویسنده در متن به فلسفه دارد، به‌صورت زیر می‌تواند خلاصه شود:

1. **انتقاد از ترکیب فلسفه‌های ساده‌سلیقه‌ای**  
   او اشاره می‌کند که «اپیکوریسم را با ایزولاسیونیسم ستد نکنم»، یعنی از ادای ستایش برای ترکیبی که به‌نظر می‌رسد فلسفه را به‌صورت جداگانه و بدون درک عمیق، به‌صورت ساده‌سلیقه‌ای می‌پذیرد، خودداری می‌کند.

2. **شکاکی نسبت به تأثیرات فلسفی**  
   با گفتن «من با علم به عوارض آن دگراندیش باشم»، نشان می‌دهد که او به‌خاطر آگاهی از عوارض و پیامدهای احتمالی فلسفه، نسبت به آن دگراندیش و محتاط است.

3. **دعوت به تفکر عمیق‌تر**  
   در پایان، او «بعد از صعدایی عمیق» و «راه زیستن را همچون پارسایی» را توصیه می‌کند؛ یعنی به جای پذیرش فلسفه به‌صورت سطحی، باید به‌صورت عمیق و با توجه به وابستگی‌ها و پیچیدگی‌های زندگی، آن را درک کرد.

به‌طور خلاصه، نویسنده فلسفه را نه‌تنها به‌عنوان یک مجموعهٔ نظری بلکه به‌عنوان یک حوزهٔ پیچیده و پر از پیامدهای ممکن می‌بیند و از تبعیت ساده‌سلیقه‌ای و ترکیب‌های ناپایدار فلسفی خودداری می‌کند.
